### Dependencies

In [ ]:
!pip -q install -U datasets tokenizers accelerate tqdm numpy einops imageio pillow transformers
!pip install hf_transfer

In [ ]:
vocab_size = 26_000
train_examples = 1000_000
val_examples = 10_000
tokenizer_train_examples = 150_000
train_steps = 20_000
batch_size = 32
grad_accumulation = 2
learning_rate = 2e-4
weight_decay = 0.1
warmup_steps = 1_000
seq_len = 256
d_model = 512
n_layers = 10
n_heads = 8
d_ff = 4 * d_model
diffusion_steps = 128

In [ ]:
import os , math, time , json, random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

from datasets import load_dataset

train_ds = load_dataset("roneneldan/TinyStories", split = f"train[:{train_examples}]")
val_ds = load_dataset("roneneldan/TinyStories", split = f"validation[:{val_examples}]")
print(train_ds,val_ds)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NKFC
from tokenizers.processors import TemplateProcessing

special_tokens = [
    "[PAD]" , "[UNK]" , ["BOS"] , ["EOS"] , "[MASK]",
    "<|user|>" , "<|assistant|>" , "<|system|>" , "<|end|>"
]

def tokenizer_training_iterator(ds,n_examples):
    for i in range(min(n_examples,len(ds))):
        story = ds[i]["text"].strip()
        yield f"<|user|>\nWrite a short story.\n<|assistant|>\n{story}\n<|end|>\n"

tokenizer = Tokenizer(BPE(unk_token=["UNK"]))
tokenizer.normalizer = NKFC()
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space= False)


trainer = BpeTrainer(
    vocab_size= 26_000,
    min_frequency=2,
    special_tokens= special_tokens
)

print("Training Tokenizer....")
tokenizer.train_from_iterator(
    tokenizer_training_iterator(train_ds,tokenizer_train_examples),
    trainer = trainer
)

bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

tokenizer.post_processor = TemplateProcessing(
    single = "[BOS] $A [EOS]",
    special_tokens= [("[BOS]",bos_id),("[EOS]",eos_id)],
)

tokenizer.decoder = ByteLevelDecoder()

tokenizer_dir = "tokenizer_from_scratch"
os.makedirs(tokenizer_dir,exist_ok=True)
tokenizer_file = os.path.join(tokenizer_dir,"tokenizer.json")
tokenizer.save(tokenizer_file)

print(f"Saved Tokenizer: f{tokenizer_file}")
print(f"Vocab Size: {tokenizer.get_vocab_size()}")

In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_file)

hf_tokenizer.pad_token  = "[PAD]"
hf_tokenizer.unk_token  = "[UNK]"
hf_tokenizer.bos_token  = "[BOS]"
hf_tokenizer.eos_token  = "[EOS]"
hf_tokenizer.mask_token = "[MASK]"

hf_tokenizer.add_special_tokens({
    "additional_special_tokens": ["<|user|>", "<|assistant|>", "<|system|>", "<|end|>"]
})

PAD_ID  = hf_tokenizer.pad_token_id
MASK_ID = hf_tokenizer.mask_token_id
BOS_ID  = hf_tokenizer.bos_token_id
EOS_ID  = hf_tokenizer.eos_token_id

print("PAD_ID:", PAD_ID, "MASK_ID:", MASK_ID, "BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID)
print("Example encoding:", hf_tokenizer.encode("Hello world!")[:20])

In [ ]:
from dataclasses import dataclass

@dataclass
class DiffusionLLMConfig:
    vocab_size: int 
    seq_len: int 
    d_model: int 
    n_layers: int
    n_heads: int
    d_ff: int 
    dropout: float
    diffusion_steps: int

class DiffusionTransformerLM(nn.Module):
    def __init__(self, cfg:DiffusionLLMConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.time_emb = nn.Embedding(cfg.diffusion_steps + 1 ,cfg.d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model = cfg.d_model,
            nhead = cfg.n_heads,
            dim_feedforward= cfg.d_ff,
            dropout= cfg.dropout,
            batch_first= cfg.dropout,
            batch_first = True,
            activation= "gelu",
            norm_first= True,
        )

        self.encoder = nn.TransformerEncoder(encoder_layer=enc_layer,num_layers=cfg.n_layers)
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model,cfg.vocab_size, bias =False)

        self.lm_head.weight = self.tok_emb.weight
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self,input_ids,timesteps,attention_mask = None):
        B,L = input_ids.shape
        if L>self.cfg.seq_len:
            raise ValueError(f"Sequence Length {L} > Configuration Sequence Length {self.cfg.seq_len}")
        
        pos = torch.arange(0,L,device = input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)

        t_emb = self.time_emb(timesteps).unsqueeze(1)
        x = x + t_emb
        x = self.drop(x)

        if attention_mask is None:
            src_key_padding_mask = None
        else:
            src_key_padding_mask = ~attention_mask
        
        x = self.encoder(x,src_key_padding_mask = src_key_padding_mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits
    
cfg = DiffusionLLMConfig(
    vocab_size= len(hf_tokenizer),
    seq_len= seq_len,
    d_model = d_model,
    n_layers= n_layers,
    n_heads = n_heads,
    d_ff = d_ff,
    dropout = 0.1,
    diffusion_steps = diffusion_steps,
)

model = DiffusionTransformerLM(cfg)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model Parameters: {n_params}")

In [ ]:
def format_as_chat(story_text:str) -> str:
    story_text = story_text.strip()
    return f"<|user|>\nWrite a short story.\n<|assistant|>\n{story_text}\n<|end|>\n"

class TokenBlockDataset(IterableDataset):
    def __init__(self,hf_ds, tokenizer, seq_len, shuffle = False, seed = 0):
        self.hf_ds = hf_ds
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.shuffle = shuffle
        self.seed = seed
    
    def __iter__(self):
        indices = list(range(len(self.hf_ds)))
        if self.shuffle:
            rng = random.Random(self.seed)
            rng.shuffle(indices)
        
        buffer = []
        for idx in indices:
            text = format_as_chat(self.hf_ds[idx]["text"])
            token_ids = self.tokenizer.encode(text,special_tokens = True)
            buffer.extend(token_ids)

            while len(buffer) >= self.seq_len:
                block = buffer[:self.seq_len]
                buffer = buffer[self.seq_len:]
                yield torch.tensor(block, dtype = torch.long)

train_blocks = TokenBlockDataset(train_ds, hf_tokenizer, seq_len, shuffle = True, seed = 42)
val_blocks = TokenBlockDataset(val_ds, hf_tokenizer, seq_len, shuffle = False)

def collate_blocks(batch):
    input_ids = torch.stack(batch,dim = 0)
    attention_mask = (input_ids != PAD_ID)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask
    }

train_loader = DataLoader(train_blocks, batch_size = batch_size, collate_fn = collate_blocks)
val_loader = DataLoader(val_blocks, batch_size = batch_size, collate_fn = collate_blocks)

b = next(iter(train_loader))
print("Batch Input IDs Shape:", b['input_ids'].shape)
print({k:v.shape for k,v in b.items()})
print(f"Decoded Snippet: {hf_tokenizer.decode(b['input_ids'][0][:120].tolist())}")


## Diffusion Corruption(Masking) + Training Loss

In [ ]:
def mask_ratio_schedule(t,T: int):
    return t.float() / float(T)

@torch.no_grad()
def corrupt_with_mask(input_ids, attention_mask, t, mask_token_id: int, T: int):
    B, L = input_ids.shape
    ratio = mask_ratio_schedule(t,T).unsqueeze(1) 
    
    can_mask = attention_mask.clone()
    can_mask = can_mask & ((input_ids != BOS_ID) & (input_ids != EOS_ID) & (input_ids != PAD_ID))

    rand = torch.rand((B,L) , device = input_ids.device)
    mask_positions = (rand < ratio) & can_mask

    noisy = input_ids.clone()
    noisy[mask_positions] = mask_token_id

    labels = torch.full_like(input_ids, fill_value = -100)
    labels[mask_positions] = input_ids[mask_positions]

    return noisy, labels  , mask_positions

def diffusion_loss(model, batch, T: int):
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    B, L = input_ids.shape
    t = torch.randint(1, T + 1, (B,) , device = input_ids.device)

    noisy_input_ids, labels, mask_positions = corrupt_with_mask(input_ids, attention_mask, t, MASK_ID, T)   
    logits = model(noisy_input_ids, t, attention_mask = attention_mask)
    loss = f.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index = -100)
    return loss


### Diffusion LLM Training

In [ ]:
from accelerate import Accelerator
from transformers import get_cosine_schedule_with_warmup

accelerator = Accelerator()
device = accelerator.device

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = weight_decay)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps = warmup_steps,
    num_training_steps = train_steps
)

model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, scheduler
)

def eval_loss(n_batches = 20):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= n_batches:
                break

            loss = diffusion_loss(model, batch, T = diffusion_steps)
            gathered = accelerator.gather(loss.detach().float().reshape(1))
            losses.append(gathered.cpu())
    model.train()

    if len(losses) == 0:
        return float('nan')
    
    losses = torch.cat(losses, dim = 0)
    return losses.mean().item()
    

model.train()
bar = tqdm(range(train_steps), disable = not accelerator.is_main_process)
running = []

train_iter = iter(train_loader)

for step in bar:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)
    
    loss = diffusion_loss(model, batch, T = diffusion_steps) / grad_accumulation
    accelerator.backward(loss)

    if (step + 1) % grad_accumulation == 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    running.append(loss.item() * grad_accumulation)

    if (step + 1) % 50 == 0 and accelerator.is_main_process:
        bar.set_description(f"Loss= {np.mean(running[-50:]:.4f)}  lr = {scheduler.get_last_lr()[0]:.2e}")
    
    if (step + 1) % 500 == 0 and accelerator.is_main_process:
        val_loss = eval_loss(n_batches = 10)
        print(f"\nStep {step + 1} Validation Loss: {val_loss:.4f}\n")
    
if accelerator.is_main_process:
    output_dir = "diffusion_lm_model"
    os.makedirs(output_dir, exist_ok = True)
    torch.save(accelerator.unwrap_model(model).state_dict(), os.path.join(output_dir, "model.pt"))
    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump(cfg.__dict__, f, indent = 2)
    hf_tokenizer.save_pretrained(os.path.join(output_dir, "tokenizer"))
    print(f"Model and Tokenizer saved to {output_dir}")


### Diffusion Sampling (Progressive UnMasking)

In [ ]:
@torch.no_grad()
def diffusion_generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 128,
    diffusion_steps: int = 64,
    temperature:float = 1.0,
    top_k: int = 0,
    record_steps: bool = True
):
    model.eval()
    device = next(model.parameters()).device

    prompt_ids = tokenizer.encode(prompt,add_special_tokens = True)
    prompt_ids = torch.tensor(prompt_ids, dtype = torch.long, device = device).unsqueeze(0)

    prompt_len = prompt_ids.size(1)
    L = min(model.cfg.seq_len, prompt_len + max_new_tokens)
    gen_len = L - prompt_len

    x  = torch.full((1,L), MASK_ID, dtype = torch.long, device = device)
    x[:, :prompt_len] = prompt_ids[:,:prompt_len]

    fixed = torch.zeros((1,L), dtype = torch.bool, device = device)
    fixed[:, :prompt_len] = True

    attention_mask = torch.ones((1,L), dtype = torch.bool, device = device)

    frames = []

    def sample_from_logits(logits):
        if temperature !=1.0:
            logits = logits / temperature
        
        if top_k and top_k > 0:
            topk_values, topk_indices = torch.topk(logits, top_k,dim=-1)
            filtered = torch.full_like(logits, float('-inf'))
            filtered.scatter_(-1, topk_indices, topk_values)
            logits = filtered

        probs = F.softmax(logits, dim=-1)
        flat = probs.view(-1, probs.size(-1))
        sampled = torch.multinomial(flat, num_samples=1).view(1,L)
        sampled_prob = probs.gather(-1, sampled.unsqueeze(-1)).squeeze(-1)
        return sampled, sampled_prob
    
    for s in range(diffusion_steps, 0, -1):
        t = torch.full([s], device = device, dtype = torch.long)
        logits = model(x,timesteps = t, attention_mask = attention_mask)
        sampled, conf = sample_from_logits(logits)

        updated_pos = ~fixed
        x[updated_pos] = sampled[updated_pos]

        next_ratio = float(s - 1) / float(diffusion_steps)
        target_masks = int(math.ceil(gen_len * next_ratio))

        gen_positions = torch.arange( device = device) >= prompt_len
        candidates = gen_positions & (~fixed[0])
        cand_idx = torch.where(candidates)[0]

        if target_masks > 0 and cand_idx.numel() > 0:
            cand_conf = conf[0,cand_idx]
            k = min(target_masks, cand_idx.numel())
            _ , low_idx = torch.topk(cand_conf, k = k, largest = False)
            remask_positions = cand_idx[low_idx]
            x[0,remask_positions] = MASK_ID
        
        if record_steps:
            decoded = tokenizer.decode(x[0].tolist())
            decoded = decoded.replace("[MASK]", "█")
            frames.append((s, decoded))

    final_decoded = tokenizer.decode(x[0].tolist())
    model.train()   
    return final_decoded, frames

def chat_prompt(user_msg: str, system_msg: str = None) -> str:
    parts = []
    if system_msg:
        parts.append(f"<|system|>\n {system_msg}\n")
    parts.append(f"<|user|>\n {user_msg}\n")
    parts.append("<|assistant|>\n")
    return "".join(parts)

text_user_prompt = "Once upon a time"
prompt_text = chat_prompt(text_user_prompt)

final_text, frames = diffusion_generate(
    model = accelerator.unwrap_model(model),
    tokenizer = hf_tokenizer,
    prompt = prompt_text,
    max_new_tokens = 128,
    diffusion_steps = diffusion_steps,
    temperature = 1.0,
    top_k = 50,
    record_steps = True
)

print(final_text)